# Epic Game Pass When? - pipeline runner

The one-stop orchestrator. To refresh the app:
1. Drop new scrape dumps into `data/raw/` (same filenames as before).
2. Run the cells top to bottom (or **Run All**).
3. Review the training metrics and the git diff, then run the push cell to ship to **dev**.

Each stage just calls a module in `pipeline/` - the logic lives there, this notebook only orchestrates.

**Prereqs:** open this from the repo root; set the `RAWG_API_KEY` environment variable for enrichment. Production is never touched here - the push goes to the `dev` branch only (prod is a separate manual merge).

In [1]:
from pipeline import config, ingest, enrich, train, deploy

print("Repo root:", config.REPO_ROOT)
print("RAWG keys found:", len(config.rawg_keys()))
config.ensure_dirs()

Repo root: i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor
RAWG keys found: 3


## 1. Ingest - raw dumps -> `data/processed/`
Parses the `data/raw/` scrape files into standardized `*_Processed.csv`.

In [2]:
ingest.run()

Processing Xbox New Data...
Processing PS New Data...
Processing Epic Text Data...
Processing Humble Bundle Text Data...
Extracted: Xbox(2166), PS(2661), Epic(87), HB(1174)
Processing complete. Wrote *_Processed.csv to i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\data\processed


{'Xbox': 2166, 'PS': 2661, 'Epic': 87, 'HB': 1174}

## 2. Enrich - RAWG fill + merge -> `data/canonical/`
Fills missing publisher/developer/release/metacritic via RAWG (and the local cache), then merges into the canonical datasets. Needs `RAWG_API_KEY` for any rows not already in the cache. Backups land in `data/backups/`.

In [3]:
enrich.run()

RAWG keys loaded: 3 (rotation enabled).
Loading local metadata cache...
Cache loaded with 4708 games.

--- Starting Data Processing ---

Processing Xbox_Processed.csv (Mode: REPLACE)
Need to fetch API for 0 games.
Replaced Xbox.csv with new snapshot (2166 rows).

Processing PS_Processed.csv (Mode: REPLACE)
Need to fetch API for 2 games.


Fetching: 100%|██████████| 2/2 [00:03<00:00,  1.82s/it]


Replaced PS.csv with new snapshot (2661 rows).

Processing Epic_Processed.csv (Mode: APPEND)
Need to fetch API for 1 games.


Fetching: 100%|██████████| 1/1 [00:02<00:00,  2.17s/it]


Appended to Epic.csv. Total rows: 573 (was 573)

Processing HB_Processed.csv (Mode: REPLACE)
Need to fetch API for 5 games.


Fetching: 100%|██████████| 5/5 [00:09<00:00,  1.91s/it]

Replaced HB.csv with new snapshot (1174 rows).


## 3. Train - `data/canonical/` -> `models/`
Trains one XGBoost model per platform. Review the MAE (days) and R2 below before shipping.

In [4]:
import pandas as pd
metrics = train.run()
pd.DataFrame(metrics)

PROCESSING: Xbox  (i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\data\canonical\Xbox.csv)
Valid training samples: 1767
Saved model_xbox.pkl (in-sample P50 MAE 370d) to i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\models
PROCESSING: PSPlus  (i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\data\canonical\PS.csv)
Valid training samples: 2171
Saved model_psplus.pkl (in-sample P50 MAE 532d) to i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\models
PROCESSING: Epic  (i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\data\canonical\Epic.csv)
Valid training samples: 531
Saved model_epic.pkl (in-sample P50 MAE 150d) to i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\models
PROCESSING: HumbleBundle  (i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\data\canonical\HB.csv)
Valid training samples: 1028
Saved model_humblebundle.pkl (in-sample P50 MAE 133d) to i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\models


,platform,status,samples,publishers,insample_p50_mae_days
0,Xbox,ok,1767,403,370.3
1,PSPlus,ok,2171,440,531.9
2,Epic,ok,531,267,150.0
3,HumbleBundle,ok,1028,412,133.1


## 5. Deploy - sync `data/canonical/` + `models/` -> `apps/backend/`
Copies the canonical CSVs and trained artifacts into the backend so the API serves the new models.

In [5]:
from pipeline import backtest
results = backtest.run()
report = pd.DataFrame(results)
failing = [r['platform'] for r in results if r.get('beats_baseline') is False]
if failing:
    print('WARNING: these models do not beat the best baseline (Phase 5 target):', failing)
report

,platform,status,samples,folds,model_mae_walkforward,baseline_global_median,baseline_publisher_median,model_mae_randomsplit,beats_baseline,improvement_vs_best_baseline_days
0,Xbox,ok,1767,4,947.3,1961.6,1702.3,541.3,True,755.0
1,PSPlus,ok,2171,4,1199.5,1676.1,1490.8,814.5,True,291.3
2,Epic,ok,531,4,654.6,972.5,1032.5,376.2,True,318.0
3,HumbleBundle,ok,1028,4,427.6,493.5,502.9,263.8,True,65.9


## 6. Review, then ship to `dev`
Inspect the working tree first. Then run the push cell to commit and push to `dev`, which auto-deploys the dev environment. Production stays a separate manual merge (`dev -> main`).

In [6]:
deploy.run()

Deployed 8 files to i:\Lyndon\AI ML\Project\Epic and Gamepass Predictor\apps\backend


{'copied': ['Epic.csv',
  'Xbox.csv',
  'PS.csv',
  'HB.csv',
  'model_xbox.pkl',
  'model_psplus.pkl',
  'model_epic.pkl',
  'model_humblebundle.pkl'],
 'missing': []}

## 5. Review, then ship to `dev`
Inspect the working tree first. Then run the push cell to commit and push to `dev`, which auto-deploys the dev environment. Production stays a separate manual merge (`dev -> main`).

In [7]:
!git status --short

 M apps/backend/Epic.csv
 M apps/backend/HB.csv
 M apps/backend/PS.csv
 M apps/backend/Xbox.csv
 M apps/backend/models/model_humblebundle.pkl
 M apps/backend/models/model_psplus.pkl
 M data/canonical/Epic.csv
 M data/canonical/HB.csv
 M data/canonical/PS.csv
 M data/canonical/Xbox.csv
 M models/model_humblebundle.pkl
 M models/model_psplus.pkl
 M run.ipynb


In [8]:
# Uncomment to ship to dev (auto-deploys dev; prod is a separate manual merge):
# !git add -A && git commit -m "data refresh: retrain models + redeploy" && git push origin dev